# 2D CNN — Mental Health Audio Classification
## NORMAL vs DEPRESI | 5-Fold StratifiedKFold CV

> **Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**

### Alur:
1. Pisahkan test set 10% (tidak disentuh selama CV)
2. 5-Fold CV pada 90% sisanya → Mean ± Std Macro F1
3. Best fold dievaluasi di test set
4. Download semua hasil

## Cell 1: Cek GPU

In [ ]:
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("PyTorch version:", torch.__version__)

## Cell 2: Upload & Extract Dataset
Upload `menthealth_colab.zip` yang sudah disiapkan.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih menthealth_colab.zip

In [ ]:
import zipfile, os
with zipfile.ZipFile("menthealth_colab.zip", "r") as z:
    z.extractall("/content/menthealth-ai")
print("Extracted:", os.listdir("/content/menthealth-ai"))

## Cell 3: Setup

In [ ]:
import sys
sys.path.insert(0, "/content/menthealth-ai/notebooks/CNN")
!pip install -q seaborn scikit-learn tqdm
print("Setup OK")

## Cell 4: Imports & Konfigurasi

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.StreamHandler()])

CLASS_NAMES  = ["NORMAL", "DEPRESI"]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
N_FOLDS      = 5
EPOCHS       = 100
BATCH_SIZE   = 32
LR           = 5e-4
MAX_LEN      = 800
TEST_SIZE    = 0.10
RANDOM_STATE = 42

PROJECT_ROOT = Path("/content/menthealth-ai")
FEATURES_DIR = PROJECT_ROOT / "data" / "features" / "spectrogram"
SPLITS_DIR   = PROJECT_ROOT / "data" / "splits"
RESULTS_DIR  = PROJECT_ROOT / "results"
MODEL_DIR    = PROJECT_ROOT / "models" / "dl" / "cnn"

for d in [RESULTS_DIR/"metrics", RESULTS_DIR/"plots",
          RESULTS_DIR/"confusion_matrix", MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Imports OK")

## Cell 5: Dataset & Data Helpers

In [ ]:
class MelSpectrogramDataset(Dataset):
    def __init__(self, samples, max_len=MAX_LEN):
        self.samples = samples
        self.labels  = [s[1] for s in samples]
        self.max_len = max_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        spec = np.load(npy_path)
        if spec.shape[1] > self.max_len:
            spec = spec[:, :self.max_len]
        else:
            spec = np.pad(spec, ((0,0),(0, self.max_len - spec.shape[1])), mode="constant")
        return torch.tensor(spec, dtype=torch.float32).unsqueeze(0), torch.tensor(label, dtype=torch.long)


def make_loader(samples, batch_size, shuffle=True, weighted=False):
    ds = MelSpectrogramDataset(samples)
    if weighted and shuffle:
        labels_t = torch.tensor(ds.labels, dtype=torch.long)
        counts   = torch.bincount(labels_t, minlength=2).float().clamp(min=1)
        weights  = 1.0 / counts[labels_t]
        weights  = weights / weights.sum()
        sampler  = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=False)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True, drop_last=False)


def load_patient_data():
    df     = pd.read_csv(SPLITS_DIR / "custom_2class_labels.csv")
    id_col = "Participant_ID" if "Participant_ID" in df.columns else "participant_id"
    if "label_depresi" in df.columns:
        p2l = dict(zip(df[id_col].astype(str),
                       df["label_depresi"].astype(int).map({0:"NORMAL",1:"DEPRESI"})))
    else:
        p2l = dict(zip(df[id_col].astype(str), df["Custom_Label"].astype(str)))

    all_files, avail = [], set()
    for cls, idx in CLASS_TO_IDX.items():
        cls_dir = FEATURES_DIR / cls
        if not cls_dir.exists(): continue
        for p in sorted(cls_dir.glob("*.npy")):
            pid = p.stem.split("_")[0]
            if pid in p2l:
                all_files.append((p, pid, idx)); avail.add(pid)

    patients = sorted(avail)
    labels   = [CLASS_TO_IDX[p2l[p]] for p in patients]
    print(f"Pasien: {len(patients)} | Files: {len(all_files)}")
    dist = {CLASS_NAMES[k]: v for k,v in sorted(Counter(labels).items())}
    print(f"Distribusi pasien: {dist}")
    return patients, labels, all_files


def split_samples(all_files, train_pids, val_pids):
    AUG = ["_noise","_pitch","_stretch","_combo"]
    train_s, val_s = [], []
    for path, pid, cls in all_files:
        stem = path.stem
        is_orig = stem.endswith("_mel") and not any(t in stem for t in AUG)
        if pid in train_pids:              train_s.append((path, cls))
        elif pid in val_pids and is_orig:  val_s.append((path, cls))
    return train_s, val_s

print("Dataset helpers OK")

## Cell 6: Model Arsitektur

In [ ]:
class MelSpectrogram2DCNN(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(4, 4), nn.Dropout2d(0.2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten     = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.fc(self.flatten(self.global_pool(self.conv2(self.conv1(x)))))


def spec_augment(specs, freq_mask=10, time_mask=30, n_masks=1):
    aug = specs.clone()
    _, _, n_mels, T = aug.shape
    for _ in range(n_masks):
        f  = torch.randint(0, freq_mask+1, (1,)).item()
        f0 = torch.randint(0, max(1, n_mels-f), (1,)).item()
        if f: aug[:,:,f0:f0+f,:] = 0
        t  = torch.randint(0, time_mask+1, (1,)).item()
        t0 = torch.randint(0, max(1, T-t), (1,)).item()
        if t: aug[:,:,:,t0:t0+t] = 0
    return aug

# Cek params
_m = MelSpectrogram2DCNN().to(device)
print(f"Total params: {sum(p.numel() for p in _m.parameters()):,}")
del _m

## Cell 7: Training Loop (per fold)

In [ ]:
def train_one_fold(train_samples, val_samples, fold_num, epochs=EPOCHS):
    train_loader = make_loader(train_samples, BATCH_SIZE, shuffle=True, weighted=True)
    val_loader   = make_loader(val_samples,   BATCH_SIZE, shuffle=False)

    model    = MelSpectrogram2DCNN(num_classes=2).to(device)
    labels_t = torch.tensor([s[1] for s in train_samples], dtype=torch.long)
    counts   = torch.bincount(labels_t, minlength=2).float().clamp(min=1)
    cw       = (1.0/counts); cw = (cw/cw.sum()*2).to(device)
    criterion  = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    warmup     = optim.lr_scheduler.LinearLR(optimizer, 0.2, 1.0, total_iters=5)
    plateau    = optim.lr_scheduler.ReduceLROnPlateau(
                     optimizer, mode="max", factor=0.5, patience=12, min_lr=1e-6)

    history = {k:[] for k in ["train_loss","val_loss","train_acc","val_acc",
               "val_macro_f1","val_recall_normal","val_recall_depresi",
               "val_precision_normal","val_precision_depresi"]}

    best_f1, best_epoch, best_state = -1.0, 0, None
    best_preds, best_lbls, no_improve, last_lr = [], [], 0, LR

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for specs, lbls in tqdm(train_loader, desc=f"Fold{fold_num} Ep{epoch+1:3d}", leave=False):
            specs, lbls = specs.to(device), lbls.to(device)
            specs = spec_augment(specs)
            optimizer.zero_grad()
            out = model(specs); loss = criterion(out, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
            _, pred = torch.max(out,1)
            total += lbls.size(0); correct += (pred==lbls).sum().item()

        e_tl = t_loss/max(1,len(train_loader))
        e_ta = 100.0*correct/max(1,total)

        # ── Val ──
        model.eval()
        v_loss, v_cor, v_tot = 0.0, 0, 0
        all_p, all_l = [], []
        with torch.no_grad():
            for specs, lbls in val_loader:
                specs, lbls = specs.to(device), lbls.to(device)
                out = model(specs); loss = criterion(out, lbls)
                v_loss += loss.item()
                _, pred = torch.max(out,1)
                v_tot += lbls.size(0); v_cor += (pred==lbls).sum().item()
                all_p.extend(pred.cpu().numpy()); all_l.extend(lbls.cpu().numpy())

        e_vl  = v_loss/max(1,len(val_loader))
        e_va  = 100.0*v_cor/max(1,v_tot)
        e_f1  = f1_score(all_l, all_p, average="macro", zero_division=0)
        prec, rec, _, _ = precision_recall_fscore_support(all_l, all_p, labels=[0,1], zero_division=0)

        for k,v in zip(history.keys(),[e_tl,e_vl,e_ta,e_va,e_f1,rec[0],rec[1],prec[0],prec[1]]):
            history[k].append(v)

        print(f"[Fold{fold_num}] Ep{epoch+1:3d} | TL:{e_tl:.4f} TA:{e_ta:.1f}% | "
              f"VL:{e_vl:.4f} VA:{e_va:.1f}% F1:{e_f1:.4f} | "
              f"Rec N:{rec[0]:.3f} D:{rec[1]:.3f}")

        if e_f1 > best_f1:
            best_f1=e_f1; best_epoch=epoch+1
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
            best_preds=list(all_p); best_lbls=list(all_l)
            no_improve=0
            print(f"  ✅ Best F1: {best_f1:.4f} @ epoch {best_epoch}")
        else:
            no_improve += 1

        if epoch < 5: warmup.step()
        else:         plateau.step(e_f1)

        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr != last_lr:
            print(f"  ⚠️ LR: {last_lr:.6f} → {new_lr:.6f}")
        last_lr = new_lr

        if no_improve >= 30:
            print(f"  Early stopping @ epoch {epoch+1}"); break

    return best_f1, best_epoch, best_state, best_preds, best_lbls, history

print("train_one_fold OK")

## Cell 8: Plot Helpers

In [ ]:
def plot_fold_history(history, fold_num):
    ep = range(1, len(history["train_loss"])+1)
    fig, axes = plt.subplots(1, 4, figsize=(22, 4))
    axes[0].plot(ep, history["train_loss"], label="Train", color="steelblue")
    axes[0].plot(ep, history["val_loss"],   label="Val",   color="tomato")
    axes[0].set_title(f"Loss — Fold {fold_num}"); axes[0].legend(); axes[0].grid(alpha=0.4)

    axes[1].plot(ep, history["train_acc"], label="Train", color="steelblue")
    axes[1].plot(ep, history["val_acc"],   label="Val",   color="tomato")
    axes[1].set_title(f"Accuracy — Fold {fold_num}"); axes[1].legend(); axes[1].grid(alpha=0.4)

    axes[2].plot(ep, history["val_macro_f1"], color="green", label="Macro F1")
    axes[2].set_title(f"Macro F1 — Fold {fold_num}"); axes[2].set_ylim(0,1.05)
    axes[2].legend(); axes[2].grid(alpha=0.4)

    axes[3].plot(ep, history["val_recall_normal"],     label="Recall N",  color="steelblue", marker="o", markersize=2)
    axes[3].plot(ep, history["val_recall_depresi"],    label="Recall D",  color="tomato",    marker="s", markersize=2)
    axes[3].plot(ep, history["val_precision_normal"],  label="Prec N",    color="steelblue", linestyle="--", alpha=0.6)
    axes[3].plot(ep, history["val_precision_depresi"], label="Prec D",    color="tomato",    linestyle="--", alpha=0.6)
    axes[3].axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    axes[3].set_title(f"Recall & Precision — Fold {fold_num}")
    axes[3].set_ylim(0,1.05); axes[3].legend(fontsize=7); axes[3].grid(alpha=0.4)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR/"plots"/f"fold_{fold_num}_curves.png", dpi=150)
    plt.show()


def plot_cv_summary(scores):
    mean_f1, std_f1 = np.mean(scores), np.std(scores)
    folds = [f"Fold {i+1}" for i in range(len(scores))]
    plt.figure(figsize=(8,5))
    bars = plt.bar(folds, scores, color="steelblue", alpha=0.8, edgecolor="black")
    plt.axhline(mean_f1, color="tomato", linestyle="--", lw=2,
                label=f"Mean={mean_f1:.4f} ± {std_f1:.4f}")
    plt.fill_between(range(len(folds)), mean_f1-std_f1, mean_f1+std_f1,
                     alpha=0.15, color="tomato")
    for bar, val in zip(bars, scores):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{val:.4f}", ha="center", fontsize=10)
    plt.title("5-Fold CV — Macro F1 per Fold")
    plt.ylabel("Macro F1"); plt.ylim(0,1.05)
    plt.legend(); plt.grid(alpha=0.4, axis="y")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/"plots"/"cv_summary.png", dpi=150)
    plt.show()


def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title); plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/"confusion_matrix"/filename, dpi=150)
    plt.show()

print("Plot helpers OK")

## Cell 9: Jalankan 5-Fold CV 🚀
> Estimasi waktu di T4 GPU: **~90 menit**

In [ ]:
# ── Load data ──
patients_list, patients_labels, all_files = load_patient_data()

# ── Pisahkan test set 10% ──
sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(sss.split(patients_list, patients_labels))

trainval_patients = [patients_list[i] for i in trainval_idx]
trainval_labels   = [patients_labels[i] for i in trainval_idx]
test_patients     = {patients_list[i] for i in test_idx}

AUG = ["_noise","_pitch","_stretch","_combo"]
test_samples = [(p,cls) for p,pid,cls in all_files
                if pid in test_patients
                and p.stem.endswith("_mel")
                and not any(t in p.stem for t in AUG)]

print(f"Test set : {len(test_samples)} samples dari {len(test_patients)} pasien")
print(f"TrainVal : {len(trainval_patients)} pasien")

# ── 5-Fold StratifiedKFold ──
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_f1_scores, fold_results = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(trainval_patients, trainval_labels), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    train_pids = {trainval_patients[i] for i in tr_idx}
    val_pids   = {trainval_patients[i] for i in val_idx}
    train_s, val_s = split_samples(all_files, train_pids, val_pids)

    tr_dist = {CLASS_NAMES[k]:v for k,v in sorted(Counter([s[1] for s in train_s]).items())}
    vl_dist = {CLASS_NAMES[k]:v for k,v in sorted(Counter([s[1] for s in val_s]).items())}
    print(f"Train: {tr_dist} | Val: {vl_dist}")

    result = train_one_fold(train_s, val_s, fold, epochs=EPOCHS)
    best_f1, best_epoch, best_state, best_preds, best_lbls, history = result

    fold_f1_scores.append(best_f1)
    fold_results.append(result)
    print(f"\nFold {fold} selesai — Best Macro F1: {best_f1:.4f} @ epoch {best_epoch}")

    # Plot & simpan per fold
    plot_fold_history(history, fold)
    plot_cm(best_lbls, best_preds,
            title=f"Confusion Matrix — Fold {fold} (F1={best_f1:.4f})",
            filename=f"cm_fold_{fold}.png")

    report = classification_report(best_lbls, best_preds,
                                   target_names=CLASS_NAMES, labels=[0,1], zero_division=0)
    print(report)
    with open(RESULTS_DIR/"metrics"/f"report_fold_{fold}.txt","w") as f:
        f.write(f"=== Fold {fold} | Best Epoch {best_epoch} | Macro F1: {best_f1:.4f} ===\n\n")
        f.write(report)

# ── Ringkasan CV ──
mean_f1, std_f1 = np.mean(fold_f1_scores), np.std(fold_f1_scores)
print(f"\n{'='*60}")
print(f"5-FOLD CV SELESAI")
print(f"Macro F1 per fold : {[f'{v:.4f}' for v in fold_f1_scores]}")
print(f"Mean Macro F1     : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"{'='*60}")

plot_cv_summary(fold_f1_scores)

with open(RESULTS_DIR/"metrics"/"cv_summary.txt","w") as f:
    f.write("=== 5-Fold StratifiedKFold CV Summary ===\n\n")
    for i,v in enumerate(fold_f1_scores,1):
        f.write(f"Fold {i}: Macro F1 = {v:.4f}\n")
    f.write(f"\nMean : {mean_f1:.4f}\nStd  : {std_f1:.4f}\n")

## Cell 10: Evaluasi di Test Set

In [ ]:
# ── Evaluasi best fold di test set ──
best_fold_idx = int(np.argmax(fold_f1_scores))
best_fold_num = best_fold_idx + 1
_, _, best_state, _, _, _ = fold_results[best_fold_idx]

print(f"Best fold: {best_fold_num} (F1={fold_f1_scores[best_fold_idx]:.4f})")
print(f"Evaluasi di test set ({len(test_samples)} samples)...")

model = MelSpectrogram2DCNN(num_classes=2).to(device)
model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
model.eval()

test_loader = DataLoader(MelSpectrogramDataset(test_samples),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_preds, test_lbls = [], []
with torch.no_grad():
    for specs, lbls in test_loader:
        _, pred = torch.max(model(specs.to(device)), 1)
        test_preds.extend(pred.cpu().numpy())
        test_lbls.extend(lbls.numpy())

test_f1 = f1_score(test_lbls, test_preds, average="macro", zero_division=0)
print(f"\nTest Set Macro F1: {test_f1:.4f}")

plot_cm(test_lbls, test_preds,
        title=f"Confusion Matrix — Test Set (F1={test_f1:.4f})",
        filename="cm_test_set.png")

test_report = classification_report(test_lbls, test_preds,
                                    target_names=CLASS_NAMES, labels=[0,1], zero_division=0)
print("\n=== TEST SET CLASSIFICATION REPORT ===")
print(test_report)

with open(RESULTS_DIR/"metrics"/"report_test_set.txt","w") as f:
    f.write(f"=== Test Set Report (Best Fold: {best_fold_num}) ===\n\n")
    f.write(f"CV Mean Macro F1 : {mean_f1:.4f} ± {std_f1:.4f}\n")
    f.write(f"Test Macro F1    : {test_f1:.4f}\n\n")
    f.write(test_report)

torch.save(best_state, MODEL_DIR/"best_model.pt")
print(f"\nModel tersimpan → {MODEL_DIR/'best_model.pt'}")
print("✅ Semua hasil tersimpan di results/")

## Cell 11: Download Semua Hasil

Akan mendownload 2 file:
- `results_cnn.zip` → semua visualisasi (plots, confusion matrix, metrics)
- `best_model.pt` → model weights terbaik

**Isi `results_cnn.zip`:**
```
plots/
  fold_1_curves.png  ... fold_5_curves.png   ← learning curves per fold
  cv_summary.png                              ← bar chart Mean±Std F1
confusion_matrix/
  cm_fold_1.png  ... cm_fold_5.png           ← CM per fold
  cm_test_set.png                             ← CM test set final
metrics/
  report_fold_1.txt  ... report_fold_5.txt   ← classification report per fold
  cv_summary.txt                              ← ringkasan Mean±Std
  report_test_set.txt                         ← hasil final test set
```

In [ ]:
import shutil, os
from google.colab import files

# ── Zip seluruh folder results (plots + metrics + confusion_matrix) ──
shutil.make_archive("/content/results_cnn", "zip", str(RESULTS_DIR))

# ── Copy model weights ──
shutil.copy(str(MODEL_DIR / "best_model.pt"), "/content/best_model.pt")

# ── Tampilkan isi yang akan didownload ──
print("=== ISI results_cnn.zip ===")
for folder in ["plots", "metrics", "confusion_matrix"]:
    folder_path = RESULTS_DIR / folder
    file_list = sorted(os.listdir(folder_path))
    print(f"\n\U0001f4c1 {folder}/")
    for fname in file_list:
        size_kb = (folder_path / fname).stat().st_size / 1024
        print(f"   {fname:50s} ({size_kb:.1f} KB)")

zip_size = os.path.getsize("/content/results_cnn.zip") / (1024 * 1024)
print(f"\nTotal ZIP size: {zip_size:.1f} MB")

# ── Download ──
print("\nMemulai download...")
files.download("/content/results_cnn.zip")  # semua plots + metrics + confusion matrix
files.download("/content/best_model.pt")    # model weights